In [0]:
# Databricks notebook source
# ══════════════════════════════════════
# GOLD — KPIs Lojas Fisicas
# Squad 3 — Arquitetura Medalhao
# Frequencia: toda segunda-feira as 5h
# ══════════════════════════════════════

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# constantes do notebook

SILVER_ITENS_PATH = f"{SILVER_BASE_PATH}physical_itens_venda_caixa"
SILVER_LOJAS_PATH = f"{SILVER_BASE_PATH}physical_lojas"

GOLD_WRITE_MODE = "overwrite"

print("Constantes configuradas:")
print(f"   SILVER_ITENS_PATH : {SILVER_ITENS_PATH}")
print(f"   SILVER_LOJAS_PATH : {SILVER_LOJAS_PATH}")
print(f"   TARGET_SCHEMA     : {TARGET_SCHEMA}")

In [0]:
# adls  

adls_options = get_adls_options()
print("Opcoes ADLS configuradas.")

In [0]:
# ler Silver itens e lojas

df_itens = read_delta(
    spark        = spark,
    path         = SILVER_ITENS_PATH,
    adls_options = adls_options
)
print(f"Silver itens : {df_itens.count():,} linhas")

df_lojas = read_delta(
    spark        = spark,
    path         = SILVER_LOJAS_PATH,
    adls_options = adls_options
)
print(f"Silver lojas : {df_lojas.count():,} linhas")

display(df_itens.limit(5))
display(df_lojas.limit(5))

In [0]:
# converter tipos necessarios para calculos

from pyspark.sql.functions import (
    col, count, sum as spark_sum,
    avg, round as spark_round,
    year, month,
    current_timestamp,
    dense_rank, lag, when,
    abs as spark_abs
)
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType

df_itens = (
    df_itens
    .withColumn("quantidade",
        col("quantidade").cast(DoubleType()))
    .withColumn("valor_total_item",
        col("valor_total_item").cast(DoubleType()))
    .withColumn("preco_unitario_registro",
        col("preco_unitario_registro").cast(DoubleType()))
    .withColumn("id_loja",
        col("id_loja").cast(IntegerType()))
    .withColumn("ano",
        col("ano").cast(IntegerType()))
    .withColumn("mes",
        col("mes").cast(IntegerType()))
)

df_lojas = (
    df_lojas
    .withColumn("id_loja",
        col("id_loja").cast(IntegerType()))
    .withColumn("peso_vendas",
        col("peso_vendas").cast(IntegerType()))
)

print("Tipos convertidos com sucesso!")

In [0]:
# ══════════════════════════════════════
# KPI 6 — Receita por produto por loja por mes
# Sell-out da loja fisica para negociacao
# com fornecedores
# ══════════════════════════════════════

GOLD_KPI6_PATH  = f"{GOLD_BASE_PATH}kpi_receita_produto_loja_mes"
GOLD_KPI6_TABLE = f"{TARGET_SCHEMA}.gold_kpi_receita_produto_loja_mes"

df_kpi6 = (
    df_itens
    .groupBy(
        "codigo_barras_produto",
        "id_loja",
        "ano",
        "mes"
    )
    .agg(
        spark_round(spark_sum("valor_total_item"), 2).alias("receita_total"),
        spark_round(spark_sum("quantidade"), 3).alias("qtd_total_vendida"),
        count("id_item_venda").alias("total_transacoes"),
        spark_round(avg("preco_unitario_registro"), 2).alias("preco_medio"),
    )
    .join(
        df_lojas.select("id_loja", "nome_loja", "estado_loja"),
        on="id_loja",
        how="left"
    )
    .withColumn("gold_processed_at", current_timestamp())
)

total_kpi6 = df_kpi6.count()
print(f"KPI 6 — Receita produto/loja/mes: {total_kpi6:,} linhas")
display(df_kpi6.limit(10))

In [0]:
# gravar dados no delta lake

write_delta(
    df           = df_kpi6,
    path         = GOLD_KPI6_PATH,
    mode         = GOLD_WRITE_MODE,
    partition_by = ["ano", "mes"],
    adls_options = adls_options
)

write_sql_table(
    df         = df_kpi6,
    table_name = GOLD_KPI6_TABLE,
    mode       = GOLD_WRITE_MODE
)

In [0]:
# ══════════════════════════════════════
# KPI 7 — Top 10 produtos por loja por trimestre
# Identificar produtos ancora da loja fisica
# ══════════════════════════════════════

GOLD_KPI7_PATH  = f"{GOLD_BASE_PATH}kpi_top10_produtos_trimestre"
GOLD_KPI7_TABLE = f"{TARGET_SCHEMA}.gold_kpi_top10_produtos_trimestre"

df_itens_trim = df_itens.withColumn(
    "trimestre",
    ((col("mes") - 1) / 3 + 1).cast(IntegerType())
)

df_por_produto_trim = (
    df_itens_trim
    .groupBy(
        "codigo_barras_produto",
        "id_loja",
        "ano",
        "trimestre"
    )
    .agg(
        spark_round(spark_sum("valor_total_item"), 2).alias("receita_total"),
        spark_round(spark_sum("quantidade"), 3).alias("qtd_total_vendida"),
    )
)

window_rank = (
    Window
    .partitionBy("id_loja", "ano", "trimestre")
    .orderBy(col("receita_total").desc())
)

df_kpi7 = (
    df_por_produto_trim
    .withColumn("rank_receita", dense_rank().over(window_rank))
    .filter(col("rank_receita") <= 10)
    .join(
        df_lojas.select("id_loja", "nome_loja", "estado_loja"),
        on="id_loja",
        how="left"
    )
    .withColumn("gold_processed_at", current_timestamp())
)

total_kpi7 = df_kpi7.count()
print(f"KPI 7 — Top 10 produtos/loja/trimestre: {total_kpi7:,} linhas")
display(df_kpi7.orderBy("id_loja", "ano", "trimestre", "rank_receita").limit(10))

In [0]:
# gravar dados no delta lake

write_delta(
    df           = df_kpi7,
    path         = GOLD_KPI7_PATH,
    mode         = GOLD_WRITE_MODE,
    partition_by = ["ano", "trimestre"],
    adls_options = adls_options
)

write_sql_table(
    df         = df_kpi7,
    table_name = GOLD_KPI7_TABLE,
    mode       = GOLD_WRITE_MODE
)

In [0]:
# ══════════════════════════════════════
# KPI 8 — Crescimento MoM da receita por loja
# Monitorar sazonalidade das lojas
# ══════════════════════════════════════

GOLD_KPI8_PATH  = f"{GOLD_BASE_PATH}kpi_mom_receita_loja"
GOLD_KPI8_TABLE = f"{TARGET_SCHEMA}.gold_kpi_mom_receita_loja"

df_receita_mes = (
    df_itens
    .groupBy("id_loja", "ano", "mes")
    .agg(
        spark_round(spark_sum("valor_total_item"), 2).alias("receita_total"),
        count("id_item_venda").alias("total_itens"),
    )
)

window_mom = (
    Window
    .partitionBy("id_loja")
    .orderBy("ano", "mes")
)

df_kpi8 = (
    df_receita_mes
    .withColumn("receita_mes_anterior",
        lag("receita_total", 1).over(window_mom))
    .withColumn(
        "crescimento_mom_pct",
        when(
            col("receita_mes_anterior").isNotNull() &
            (col("receita_mes_anterior") > 0),
            spark_round(
                (col("receita_total") - col("receita_mes_anterior")) /
                col("receita_mes_anterior") * 100,
                2
            )
        ).otherwise(None)
    )
    .join(
        df_lojas.select("id_loja", "nome_loja", "estado_loja"),
        on="id_loja",
        how="left"
    )
    .withColumn("gold_processed_at", current_timestamp())
)

total_kpi8 = df_kpi8.count()
print(f"KPI 8 — MoM receita por loja: {total_kpi8:,} linhas")
display(df_kpi8.orderBy("id_loja", "ano", "mes").limit(10))

In [0]:
# gravar dados no delta lake

write_delta(
    df           = df_kpi8,
    path         = GOLD_KPI8_PATH,
    mode         = GOLD_WRITE_MODE,
    partition_by = ["ano", "mes"],
    adls_options = adls_options
)

write_sql_table(
    df         = df_kpi8,
    table_name = GOLD_KPI8_TABLE,
    mode       = GOLD_WRITE_MODE
)

In [0]:
# ══════════════════════════════════════
# KPI 9 — Ticket medio em itens por loja
# Lojas com ticket mais alto em itens
# indicam perfil de compra diferente
# ══════════════════════════════════════

GOLD_KPI9_PATH  = f"{GOLD_BASE_PATH}kpi_ticket_medio_loja"
GOLD_KPI9_TABLE = f"{TARGET_SCHEMA}.gold_kpi_ticket_medio_loja"

df_por_transacao = (
    df_itens
    .groupBy("id_transacao", "id_loja", "ano", "mes")
    .agg(
        spark_round(spark_sum("valor_total_item"), 2).alias("valor_transacao"),
        count("id_item_venda").alias("qtd_itens_transacao"),
    )
)

df_kpi9 = (
    df_por_transacao
    .groupBy("id_loja", "ano", "mes")
    .agg(
        spark_round(avg("valor_transacao"), 2).alias("ticket_medio_valor"),
        spark_round(avg("qtd_itens_transacao"), 2).alias("ticket_medio_itens"),
        count("id_transacao").alias("total_transacoes"),
    )
    .join(
        df_lojas.select(
            "id_loja", "nome_loja",
            "cidade_loja", "estado_loja", "peso_vendas"
        ),
        on="id_loja",
        how="left"
    )
    .withColumn("gold_processed_at", current_timestamp())
)

total_kpi9 = df_kpi9.count()
print(f"KPI 9 — Ticket medio por loja: {total_kpi9:,} linhas")
display(df_kpi9.orderBy("ano", "mes", "id_loja").limit(10))

In [0]:
# gravar dados no delta lake

write_delta(
    df           = df_kpi9,
    path         = GOLD_KPI9_PATH,
    mode         = GOLD_WRITE_MODE,
    partition_by = ["ano", "mes"],
    adls_options = adls_options
)

write_sql_table(
    df         = df_kpi9,
    table_name = GOLD_KPI9_TABLE,
    mode       = GOLD_WRITE_MODE
)

In [0]:
# ══════════════════════════════════════
# KPI 4 — Receita total por loja por ano
# Identificar lojas mais rentaveis
# para decisao de investimento
# ══════════════════════════════════════

GOLD_KPI4_PATH  = f"{GOLD_BASE_PATH}kpi_receita_loja_ano"
GOLD_KPI4_TABLE = f"{TARGET_SCHEMA}.gold_kpi_receita_loja_ano"

df_kpi4_base = (
    df_itens
    .groupBy("id_loja", "ano")
    .agg(
        spark_round(spark_sum("valor_total_item"), 2).alias("receita_total_ano"),
        count("id_item_venda").alias("total_itens"),
        count(col("id_transacao")).alias("total_transacoes"),
    )
    .join(
        df_lojas.select(
            "id_loja", "nome_loja",
            "cidade_loja", "estado_loja", "peso_vendas"
        ),
        on="id_loja",
        how="left"
    )
)

window_ranking = (
    Window
    .partitionBy("ano")
    .orderBy(col("receita_total_ano").desc())
)

df_kpi4 = (
    df_kpi4_base
    .withColumn("ranking_receita", dense_rank().over(window_ranking))
    .withColumn("gold_processed_at", current_timestamp())
)

total_kpi4 = df_kpi4.count()
print(f"KPI 4 — Receita loja/ano: {total_kpi4:,} linhas")
display(df_kpi4.orderBy("ano", "ranking_receita").limit(10))

In [0]:
# gravar dados no delta lake

write_delta(
    df           = df_kpi4,
    path         = GOLD_KPI4_PATH,
    mode         = GOLD_WRITE_MODE,
    partition_by = ["ano"],
    adls_options = adls_options
)

write_sql_table(
    df         = df_kpi4,
    table_name = GOLD_KPI4_TABLE,
    mode       = GOLD_WRITE_MODE
)

In [0]:
# ══════════════════════════════════════
# KPI 5 — Crescimento YoY por loja
# Comparar desempenho de cada loja
# ano a ano
# ══════════════════════════════════════

GOLD_KPI5_PATH  = f"{GOLD_BASE_PATH}kpi_crescimento_yoy_loja"
GOLD_KPI5_TABLE = f"{TARGET_SCHEMA}.gold_kpi_crescimento_yoy_loja"

window_yoy = (
    Window
    .partitionBy("id_loja")
    .orderBy("ano")
)

df_kpi5 = (
    df_kpi4
    .select(
        "id_loja", "nome_loja",
        "estado_loja", "ano",
        "receita_total_ano"
    )
    .withColumn("receita_ano_anterior",
        lag("receita_total_ano", 1).over(window_yoy))
    .withColumn(
        "crescimento_yoy_pct",
        when(
            col("receita_ano_anterior").isNotNull() &
            (col("receita_ano_anterior") > 0),
            spark_round(
                (col("receita_total_ano") - col("receita_ano_anterior")) /
                col("receita_ano_anterior") * 100,
                2
            )
        ).otherwise(None)
    )
    .withColumn("gold_processed_at", current_timestamp())
)

total_kpi5 = df_kpi5.count()
print(f"KPI 5 — Crescimento YoY por loja: {total_kpi5:,} linhas")
display(df_kpi5.orderBy("id_loja", "ano").limit(10))

In [0]:
# gravar dados no delta lake

write_delta(
    df           = df_kpi5,
    path         = GOLD_KPI5_PATH,
    mode         = GOLD_WRITE_MODE,
    partition_by = ["ano"],
    adls_options = adls_options
)

write_sql_table(
    df         = df_kpi5,
    table_name = GOLD_KPI5_TABLE,
    mode       = GOLD_WRITE_MODE
)

In [0]:
# ══════════════════════════════════════
# KPI 6L — Numero de transacoes por loja por mes
# Volume operacional para planejamento
# de equipe e infraestrutura
# ══════════════════════════════════════

GOLD_KPI6L_PATH  = f"{GOLD_BASE_PATH}kpi_transacoes_loja_mes"
GOLD_KPI6L_TABLE = f"{TARGET_SCHEMA}.gold_kpi_transacoes_loja_mes"

df_kpi6l = (
    df_itens
    .select("id_transacao", "id_loja", "ano", "mes")
    .distinct()
    .groupBy("id_loja", "ano", "mes")
    .agg(
        count("id_transacao").alias("total_transacoes")
    )
    .join(
        df_lojas.select(
            "id_loja", "nome_loja",
            "cidade_loja", "estado_loja"
        ),
        on="id_loja",
        how="left"
    )
    .withColumn("gold_processed_at", current_timestamp())
)

total_kpi6l = df_kpi6l.count()
print(f"KPI 6L — Transacoes/loja/mes: {total_kpi6l:,} linhas")
display(df_kpi6l.orderBy("ano", "mes", "id_loja").limit(10))

In [0]:
# gravar dados no delta lake

write_delta(
    df           = df_kpi6l,
    path         = GOLD_KPI6L_PATH,
    mode         = GOLD_WRITE_MODE,
    partition_by = ["ano", "mes"],
    adls_options = adls_options
)

write_sql_table(
    df         = df_kpi6l,
    table_name = GOLD_KPI6L_TABLE,
    mode       = GOLD_WRITE_MODE
)

In [0]:
# ══════════════════════════════════════
# KPI 8L — Ticket medio por loja por semestre
# Comparar ticket medio entre lojas
# grandes e pequenas
# ══════════════════════════════════════

GOLD_KPI8L_PATH  = f"{GOLD_BASE_PATH}kpi_ticket_medio_semestre"
GOLD_KPI8L_TABLE = f"{TARGET_SCHEMA}.gold_kpi_ticket_medio_semestre"

df_itens_sem = df_itens.withColumn(
    "semestre",
    when(col("mes") <= 6, 1).otherwise(2)
)

df_por_transacao_sem = (
    df_itens_sem
    .groupBy("id_transacao", "id_loja", "ano", "semestre")
    .agg(
        spark_round(spark_sum("valor_total_item"), 2).alias("valor_transacao"),
        count("id_item_venda").alias("qtd_itens"),
    )
)

df_kpi8l = (
    df_por_transacao_sem
    .groupBy("id_loja", "ano", "semestre")
    .agg(
        spark_round(avg("valor_transacao"), 2).alias("ticket_medio_valor"),
        spark_round(avg("qtd_itens"), 2).alias("ticket_medio_itens"),
        count("id_transacao").alias("total_transacoes"),
    )
    .join(
        df_lojas.select(
            "id_loja", "nome_loja",
            "cidade_loja", "estado_loja"
        ),
        on="id_loja",
        how="left"
    )
    .withColumn("gold_processed_at", current_timestamp())
)

total_kpi8l = df_kpi8l.count()
print(f"KPI 8L — Ticket medio/loja/semestre: {total_kpi8l:,} linhas")
display(df_kpi8l.orderBy("id_loja", "ano", "semestre").limit(10))

In [0]:
# gravar dados no delta lake

write_delta(
    df           = df_kpi8l,
    path         = GOLD_KPI8L_PATH,
    mode         = GOLD_WRITE_MODE,
    partition_by = ["ano", "semestre"],
    adls_options = adls_options
)

write_sql_table(
    df         = df_kpi8l,
    table_name = GOLD_KPI8L_TABLE,
    mode       = GOLD_WRITE_MODE
)

In [0]:
# ══════════════════════════════════════
# KPI 10 — Validar todas as lojas na Gold
# Loja ausente na Gold causa dados
# faltantes nas analises
# ══════════════════════════════════════

print("=" * 55)
print("KPI 10 — VALIDACAO LOJAS NA GOLD")
print("=" * 55)

ids_lojas_silver = df_lojas.select("id_loja").distinct()
ids_lojas_gold   = df_kpi4.select("id_loja").distinct()

lojas_faltantes = ids_lojas_silver.subtract(ids_lojas_gold).count()

if lojas_faltantes > 0:
    print(f"ATENCAO: {lojas_faltantes} lojas ausentes na Gold!")
    ids_lojas_silver.subtract(ids_lojas_gold).show()
else:
    print(
        f"Validacao OK: todas as "
        f"{ids_lojas_silver.count():,} lojas "
        f"estao presentes na Gold."
    )